# Feature engineering — Renewal Calls
Small and clear steps to build per-customer call features.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../../data/02_processed/processed_renewal_calls.csv")

# clean column names
df.columns = df.columns.str.lower().str.strip()

df.head()

,call_id,call_direction,co_ref,call_date,agent_renewal_pitch_category,customer_renewal_response_category,agent_response_category,membership_renewal_decision,serious_complaint,other_complaint,...,percentage_price_increase_mentioned,monetary_price_increase_mentioned,price_range_mentioned,customer_asked_for_justification,customer_response,desire_to_cancel,discount_offered,analysed_call,call_number,call_year
0,5.950000e+11,Outbound,UB0899,29-01-2025,Discussion / Introduction / Inquiry,Discount and Offer,Discount and Offer,No,No,No,...,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,1.0,3,2025
1,5.970000e+11,OUT_BOUND,HN5141,26-02-2025,Price and Cost,Agreement,Customer Communication,No,No,No,...,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,1.0,2,2025
2,5.950000e+11,Outbound,BP5009,24-01-2025,Expiration / Due,Agreement,Accreditation and Certification,No,No,No,...,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,1.0,1,2025
3,6.520000e+11,OUT_BOUND,XP8119,09-06-2025,Auto / Automatic,Agreement,Accreditation and Certification,No,No,No,...,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,1.0,1,2025
4,5.370000e+11,Outbound,ZL7978,20-08-2024,Unknown,Unknown,Unknown,No,No,No,...,No,No,Not Discussed,No,Not Discussed,Not Discussed,No,1.0,28,2024


In [3]:
print(df.columns.tolist())

['call_id', 'call_direction', 'co_ref', 'call_date', 'agent_renewal_pitch_category', 'customer_renewal_response_category', 'agent_response_category', 'membership_renewal_decision', 'serious_complaint', 'other_complaint', 'discussion_on_price_increase', 'renewal_impact_due_to_price_increase', 'discount_or_waiver_requested', 'call_reschedule_request', 'agent_flagged_membership_status_alert', 'agent_renewal_initiation', 'explicit_competitor_mention', 'explicit_switching_intent', 'mentioned_competitors', 'price_switching_mentioned', 'competitor_value_comparison', 'competitor_benefits_mentioned', 'topic_introduced_by', 'percentage_price_increase_mentioned', 'monetary_price_increase_mentioned', 'price_range_mentioned', 'customer_asked_for_justification', 'customer_response', 'desire_to_cancel', 'discount_offered', 'analysed_call', 'call_number', 'call_year']


In [4]:
df['call_date'] = pd.to_datetime(df['call_date'], errors='coerce')

/tmp/ipykernel_60030/1835517686.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['call_date'] = pd.to_datetime(df['call_date'], errors='coerce')


In [5]:
bill_df = pd.read_csv("../../data/02_processed/processed_billings.csv")
bill_df.columns = bill_df.columns.str.lower().str.strip()

bill_df['prospect_renewal_date'] = pd.to_datetime(
    bill_df['prospect_renewal_date'], errors='coerce'
)

renewal_map = bill_df[['co_ref', 'prospect_renewal_date']].drop_duplicates()

df = df.merge(renewal_map, on='co_ref', how='left')

/tmp/ipykernel_60030/3797275103.py:1: DtypeWarning: Columns (10,11,14,15,23,48) have mixed types. Specify dtype option on import or set low_memory=False.
  bill_df = pd.read_csv("../../data/02_processed/processed_billings.csv")


In [6]:
df = df.dropna(subset=['call_date', 'prospect_renewal_date'])

In [7]:
df['cutoff_date'] = df['prospect_renewal_date'] - pd.Timedelta(days=14)

In [8]:
df = df[df['call_date'] <= df['cutoff_date']]

In [9]:
df = df.sort_values(by=['co_ref', 'call_date'])

def to_binary_flag(series, pattern):
    s = series.astype(str).str.strip().str.lower()
    # Remove markdown wrappers and common noise before pattern matching.
    s = s.str.replace(r'[\[\]\*]', '', regex=True)
    return s.str.contains(pattern, regex=True, na=False).astype(int)

flag_patterns = {
    'serious_complaint': r'\byes\b',
    'explicit_switching_intent': r'\byes\b|switch',
    'desire_to_cancel': r'\byes\b|desire\s*to\s*cancel|desired\s*to\s*cancel|cancel',
    'discussion_on_price_increase': r'\byes\b',
    'discount_or_waiver_requested': r'\byes\b|discount\s*code'
}

for col, pattern in flag_patterns.items():
    if col in df.columns:
        df[col] = to_binary_flag(df[col], pattern)

In [10]:
agg_df = df.groupby('co_ref').agg({
    'call_date': ['count', 'max']
}).reset_index()

agg_df.columns = ['co_ref', 'total_calls', 'last_call_date']

In [11]:
cutoff_map = df[['co_ref', 'cutoff_date']].drop_duplicates()

agg_df = agg_df.merge(cutoff_map, on='co_ref', how='left')

agg_df['days_since_last_call'] = (
    agg_df['cutoff_date'] - agg_df['last_call_date']
).dt.days

In [12]:
df['days_before_cutoff'] = (df['cutoff_date'] - df['call_date']).dt.days

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 7].groupby('co_ref')['call_id'].count().rename('calls_last_7'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 14].groupby('co_ref')['call_id'].count().rename('calls_last_14'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 30].groupby('co_ref')['call_id'].count().rename('calls_last_30'),
    on='co_ref', how='left'
)

In [13]:
agg_df = agg_df.merge(
    df.groupby('co_ref')['serious_complaint'].sum().rename('serious_complaints'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['explicit_switching_intent'].sum().rename('switch_intent'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['desire_to_cancel'].sum().rename('cancel_intent'),
    on='co_ref', how='left'
)

In [14]:
agg_df = agg_df.merge(
    df.groupby('co_ref')['discussion_on_price_increase'].sum().rename('price_discussions'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['discount_or_waiver_requested'].sum().rename('discount_requests'),
    on='co_ref', how='left'
)

In [15]:
agg_df['recent_call_ratio'] = agg_df['calls_last_7'] / (agg_df['total_calls'] + 1)

In [16]:
agg_df = agg_df.fillna(0)

In [17]:
agg_df.to_csv("../../data/03_final/final_renewal_calls_features.csv", index=False)